FastAPI 是目前 Python 领域最快、最现代的 Web 框架之一。它的核心优势在于利用 Python 的 **类型提示 (Type Hints)** 来实现自动化的数据校验和文档生成。

以下是 FastAPI 基本语法的核心总结和代码示例。

---

## 1. 基础结构与启动
创建一个 FastAPI 实例，并定义一个简单的路由。

```python
from fastapi import FastAPI

# 1. 初始化应用
app = FastAPI()

# 2. 定义路径操作装饰器
@app.get("/")
async def root():
    # 3. 返回内容（会自动转换为 JSON）
    return {"message": "Hello World"}
```
> **启动方式**：在终端运行 `uvicorn main:app --reload`

---

## 2. 路径参数 (Path Parameters)
用于捕获 URL 路径中的动态部分。

```python
@app.get("/items/{item_id}")
async def read_item(item_id: int): # 自动进行类型转换和校验
    return {"item_id": item_id}
```

---

## 3. 查询参数 (Query Parameters)
不在路径中定义的函数参数，会自动被识别为查询参数（URL 中 `?` 后面的部分）。

```python
@app.get("/users/")
async def read_users(skip: int = 0, limit: int = 10):
    # 访问示例：/users/?skip=20&limit=5
    return {"skip": skip, "limit": limit}
```

---

## 4. 请求体 (Request Body)
使用 **Pydantic** 模型来定义请求体。FastAPI 会自动解析 JSON 并验证数据。

```python
from pydantic import BaseModel
from typing import Optional

# 定义数据模型
class Item(BaseModel):
    name: str
    description: Optional[str] = None
    price: float
    tax: float = 10.5

@app.post("/items/")
async def create_item(item: Item):
    return {"item_name": item.name, "total_price": item.price + item.tax}
```

---

## 5. 数据校验 (Query & Path Validations)
使用 `Query`, `Path`, `Body` 来添加额外的校验规则。

```python
from fastapi import Query, Path

@app.get("/search/")
async def search(
    q: str = Query(None, min_length=3, max_length=50), # 字符串长度限制
    page: int = Query(1, gt=0) # 大于 0
):
    return {"q": q}
```

---

## 6. 响应模型 (Response Model)
通过 `response_model` 参数，可以过滤输出数据，防止敏感信息泄露。

```python
from pydantic import BaseModel, EmailStr

class UserIn(BaseModel):
    username: str
    password: str
    email: EmailStr

class UserOut(BaseModel):
    username: str
    email: EmailStr

@app.post("/user/", response_model=UserOut)
async def create_user(user: UserIn):
    # 虽然接收了 password，但返回时只会包含 UserOut 定义的字段
    return user
```

---

## 7. 依赖注入 (Dependency Injection)
这是 FastAPI 最强大的功能之一，用于共享逻辑（如数据库连接、权限验证）。

```python
from fastapi import Depends

# 模拟依赖函数
async def common_parameters(q: str = None, skip: int = 0, limit: int = 10):
    return {"q": q, "skip": skip, "limit": limit}

@app.get("/items/")
async def read_items(commons: dict = Depends(common_parameters)):
    return commons
```

---

## 8. 状态码与异常处理
手动控制返回的状态码或抛出 HTTP 错误。

```python
from fastapi import HTTPException, status

@app.get("/items/{item_id}")
async def get_item(item_id: str):
    if item_id != "foo":
        raise HTTPException(
            status_code=status.HTTP_404_NOT_FOUND, 
            detail="Item not found"
        )
    return {"item": "bar"}
```

---

## 总结：FastAPI 的核心逻辑流


| 功能 | 实现方式 | 备注 |
| :--- | :--- | :--- |
| **路由** | `@app.get()`, `@app.post()` | 支持 RESTful 所有谓词 |
| **参数提取** | 函数参数名 | 自动从路径、查询参数、Header 中提取 |
| **数据建模** | `Pydantic` | 负责校验、序列化和文档生成 |
| **自动文档** | `/docs` 或 `/redoc` | 自动生成交互式 Swagger UI |
| **并发** | `async def` | 基于 Starlette，支持高性能异步 |